In [2]:
import numpy as np
import matplotlib.pyplot as plt
import re
import h5py
import ase.io as aio
import os

In [3]:
def get_initial_info(trj):
    pattern= r"TIMESTEP"
    header=8
    N_H=0
    N_O=0
    N_Fe=0
    #SCV structure (1uc)
    N_OS_uc=8    # number oxygen in first outer layer/uc
    N_FeS_uc=4   # number of iron in first outer layer/uc
    N_FeS2_uc=3  # number of iron  second/outer layer/uc
    with open(trj, 'r') as file:

        for count, line in enumerate(file):     
            if  re.search(pattern, line) and count !=0 :
                break
                
            split_line=line.split()   
            if count == 3:
                N_atom=float(split_line[0])
            if count == 5:
                L_x=float(split_line[1])
            if count == 7:
                L_z=float(split_line[1])
            #position block
            if count > header:        
                if split_line[1] == '1':
                   N_H+=1
                if split_line[1] == '2':
                   N_O+=1   
                if split_line[1] == '3':
                   N_Fe+=1   
        L_y=L_x
        
        #replica
        for n in range(1,10):
            if 8 < L_x/n < 9:
                n_replica=n
                
        if N_atom != N_H+N_O+N_Fe:
            print("ATOM LOST")
        
        f=n_replica**2
        N_water=int(N_H/2)
        N_Fe_uc=int(N_Fe/f)
        N_O_uc=int((N_O-N_water)/f)
        w_uc=float(N_water/f/2)
        N_OS=N_OS_uc*f
        N_FeS=N_FeS_uc*f
        N_FeS2= N_FeS2_uc*f
        diag_lenght=np.sqrt(L_x**2+L_y**2)
        print("L_x = L_y = ", L_x)
        print("L_z = ", L_z)
        print("Diagonal= ", diag_lenght)
        print("N_O_uc = ",  N_O_uc)
        print("N_Fe_uc =",  N_Fe_uc)
        print("w_uc =",  w_uc)
        print(f"Size factor f = {f}")
        print(f"N_water = {N_water}")
        print(f"N_H = {N_H}")
        print(f"N_O = {N_O}")
        print(f"N_Fe = {N_Fe}")
        print(f"N_atom = {N_atom}")
        print(f"N_O surface = {N_OS}")
        print(f"N_Fe surface = {N_FeS}")
        print(f"N_Fe surface2 = {N_FeS2}")
        
        return(f, L_x, L_y, L_z, diag_lenght, N_water,  N_H, N_O, N_Fe, N_OS, N_FeS, N_FeS2)

In [4]:


def  make_folder(folder_name):
     if not os.path.exists(folder_name):
        os.mkdir(folder_name)
        
def make_empty_folder(folder_name):
    if not os.path.exists(folder_name):
        os.mkdir(folder_name)
    else:
        try:
            # List all files in the directory
            files = os.listdir(folder_name)

            # Iterate over each file and remove it
            for file in files:
                file_path = os.path.join(folder_name, file)
                if os.path.isfile(file_path):
                    os.remove(file_path)
                    #print(f"File '{file_path}' removed successfully.")
        except OSError as e:
            print(f"Error: {e.strerror}")
               
def create_file(filename):
    with open(filename, 'w'):
        pass 

def find_files_by_name(folder, file_name):
    found_files = []
    for root, dirs, files in os.walk(folder):
        for file in files:
            if file == file_name:
                found_files.append(os.path.join(root, file))
                
    return list(np.sort(found_files))



def find_files(start_dir, file_name):
    results = []
    for dirpath, _, filenames in os.walk(start_dir):
        if file_name in filenames:
            results.append(os.path.join(dirpath, file_name))
    return results

def filter_paths(paths, target_folder):
    return [path for path in paths if target_folder in os.path.normpath(path).split(os.path.sep)]

In [5]:

def get_final_positions(all_files, pattern, header):
    print("Getting final structures")
    file_count=0
    tmp_structure=[]
    final_positions=[]
    struct_count=0
    atom_count=0
    tmp_positions=[]

    # Loop through each file
    for filename in all_files:
        print("Working with:", filename)
        #OPEN FILE
        with open(filename, 'r') as file:
            # Read the content of the trajectory

            struct_count=0
            tmp_bool=True
            tmp_bool_2=True
            for global_count, line in enumerate(file):
                #NEW STRUCTURE
                if  re.search(pattern, line):
                    line_count=0
                    struct_count+=1

                    tmp_structure=tmp_positions
                    tmp_positions=[]
                #line info known  
                if struct_count == 2 and file_count ==0:
                    if tmp_bool:
                        print("N atoms = ", atom_count)
                        end_line=header+atom_count
                        print("N_lines= ", end_line) 
                        tmp_bool=False    

                #position block
                if line_count >= header: 
                    split_line=line.split()
                    tmp_positions.append([float(split_line[2]),float(split_line[3]), float(split_line[4])])

                    if struct_count == 1 and  file_count==0:    
                        atom_count+=1

                #NEW FILE
                if struct_count == 1:                       
                    if tmp_bool_2 and file_count !=0:
                         final_positions.append(tmp_structure)
                        
                         print("Saving last structure:", tmp_structure)
                         tmp_bool_2=False

                if 'end_line' in locals() or 'end_line' in globals():  
                    if line_count == end_line - 1:
                        #print("end of a structure: cleaning structure")
                        tmp_structure=[]         
                line_count+=1
        file_count+=1    
        
            
    return np.array(final_positions)
    

In [19]:

def get_initial_positions(all_files, pattern, header):
    print("Getting initial structures")
    file_count=0

    initial_positions=[]
    struct_count=0

    tmp_positions=[]

    # Loop through each file
    for filename in all_files:
        print("Working with:", filename)
        #OPEN FILE
        with open(filename, 'r') as file:
            # Read the content of the trajectory
            struct_count=0
            for global_count, line in enumerate(file):
                #NEW STRUCTURE
                if  re.search(pattern, line):
                    line_count=0
                    struct_count+=1
                    
                if struct_count == 2:
                    print("Struct == 2, saving first struct and breaking", tmp_positions )
                    initial_positions.append(tmp_positions)
                    tmp_positions=[]
                      
                    break
                #position block
                if line_count >= header: 
                    split_line=line.split()
                    tmp_positions.append([float(split_line[2]),float(split_line[3]), float(split_line[4])])

                line_count+=1
        file_count+=1    
        
                            #the first initial config doesn't matter
    return np.array(initial_positions[1:])
    

In [20]:

def get_final_displacements(initial_positions, final_positions):
    final_displacements = []
    sum_finals = np.zeros(np.array(final_positions).shape[1:])
    final_displacements.append(np.array(sum_finals.copy()))  # Copy the initial sum
    
    for disp in np.array(final_positions- initial_positions):
        sum_finals += disp
       
        final_displacements.append(np.array(sum_finals.copy()))  # Copy the updated sum
    
    
    print("Initial =, ",initial_positions   )
    print("Final",    final_positions )
    print("Displacements = ", final_positions- initial_positions)
    print("Sum of displacements (final_displ)", np.array(final_displacements))
    return np.array(final_displacements)


In [30]:


def write_all_trj(new_trj_file, all_reduced_files, time_interval, pattern, header , final_displacments  ):
    create_file(new_trj_file)
    print("WRITING")
    t=0
    file_count=0
    # Loop through each file
    line_count=0
    with open(new_trj_file, 'a') as append_file:
        for filename in all_reduced_files:
            #OPEN FILE
            print(filename, file_count)
            with open(filename, 'r') as file:
                # Read the content of the file


                for global_count, line in enumerate(file):

                    #NEW STRUCTURE
                    if  re.search(pattern, line):
                        append_file.write(line)
                        #reset line count per structure
                        line_count=0
                        atom_count=0

                    #timestep
                    elif line_count == 1:          
                            append_file.write(str(t)+'\n')
                            t+=time_interval
                    #header
                    elif line_count<header:

                            append_file.write(line)
                    #position block
                    elif line_count >= header: 

                        split_line=line.split()

                        old_position=np.array([float(split_line[2]),float(split_line[3]), float(split_line[4])])
                        delta_position=np.array(final_displacments[file_count][2]) #####TRANSLATE ONLY THE OXYGENS!!!! other wise water molecules split or [atom_count] to translate all
                        new_position=old_position+delta_position
                        new_line = split_line[0]+ '  ' +split_line[1]+ '  '+' '.join(map(str, new_position) ) + '\n' 
                        append_file.write(new_line)     
                        atom_count+=1

                    line_count+=1
            file_count+=1            

    

In [31]:


#MERGING FILES:

path_run='/data/home/salvatorer26/projects/ferrite_oxide/production/production_runs/'
starting_folder_=path_run+'water_1_no_rep_T300/'
name_file_u='W_u.dump'
name_file_d='W_d.dump'
files_u=reduced_up_files= find_files_by_name(starting_folder_, name_file_u)
files_d=reduced_up_files= find_files_by_name(starting_folder_, name_file_d)

all_reduced_files=files_d

header=9

pattern='TIMESTEP'
time_interval=500

new_trj_file='trj_all_1w_d.dump'




inital_positions=get_initial_positions(all_reduced_files, pattern, header )
final_positions=get_final_positions(all_reduced_files, pattern, header )
final_displacements=get_final_displacements(inital_positions, final_positions)


write_all_trj(new_trj_file, all_reduced_files, time_interval, pattern, header , final_displacements  )




Getting initial structures
Working with: /data/home/salvatorer26/projects/ferrite_oxide/production/production_runs/water_1_no_rep_T300/run_1/trajectory_dump_files/W_d.dump
Struct == 2, saving first struct and breaking [[6.53403, 6.96055, 6.22653], [6.52341, 5.45523, 6.40056], [6.13581, 6.27489, 6.72929]]
Working with: /data/home/salvatorer26/projects/ferrite_oxide/production/production_runs/water_1_no_rep_T300/run_2/trajectory_dump_files/W_d.dump
Struct == 2, saving first struct and breaking [[7.43615, 5.82224, 6.51396], [5.87316, 5.83203, 6.17068], [6.56803, 6.26576, 6.68562]]
Working with: /data/home/salvatorer26/projects/ferrite_oxide/production/production_runs/water_1_no_rep_T300/run_3/trajectory_dump_files/W_d.dump
Struct == 2, saving first struct and breaking [[6.38644, 5.14726, 6.53503], [5.24788, 6.25934, 6.66134], [6.19459, 6.10635, 6.67503]]
Working with: /data/home/salvatorer26/projects/ferrite_oxide/production/production_runs/water_1_no_rep_T300/run_4/trajectory_dump_files/

In [11]:
final_displacements

array([[[ 0.     ,  0.     ,  0.     ],
        [ 0.     ,  0.     ,  0.     ],
        [ 0.     ,  0.     ,  0.     ]],

       [[ 0.62489, -7.13722, -0.5059 ],
        [-1.31077, -9.36507,  0.3493 ],
        [-0.40295, -8.36714,  0.0515 ]],

       [[ 0.39813, -4.63306, -0.7795 ],
        [-1.45898, -9.29564,  0.1799 ],
        [-0.51708, -7.7774 , -0.0559 ]],

       [[ 7.50045,  2.81951, -1.6487 ],
        [ 7.96907,  0.11309,  0.0459 ],
        [ 7.68296,  0.60216, -0.3454 ]]])

In [17]:

header=9

pattern='TIMESTEP'
time_interval=500

all_tmp_file = ['./cut_tmp_1', './cut_tmp_2', './cut_tmp_3']

final_positions=get_final_positions(all_tmp_file, pattern, header )

initial_positions=get_initial_positions(all_tmp_file, pattern, header)


final_displacments=get_final_displacements(initial_positions, final_positions)

Getting final structures
Working with: ./cut_tmp_1
N atoms =  3
N_lines=  12
Working with: ./cut_tmp_2
Saving last structure: [[1.52282, 6.01339, 6.80915], [1.55321, 4.84137, 7.09866], [1.62041, 5.74217, 6.67012]]
Working with: ./cut_tmp_3
Saving last structure: [[2.74661, 2.11722, 2.11231], [2.61402, 2.20493, 2.55552], [2.8486, 2.15351, 2.77666]]
Getting initial structures
Working with: ./cut_tmp_1
Struct == 2, saving first struct and breaking [[1.53403, 6.96055, 6.22653], [6.52341, 5.45523, 6.40056], [6.13581, 6.27489, 6.72929]]
Working with: ./cut_tmp_2
Struct == 2, saving first struct and breaking [[2.62822, 5.3249, 7.14642], [6.32493, 4.60408, 6.66067], [6.80794, 5.48752, 6.60977]]
Working with: ./cut_tmp_3
Struct == 2, saving first struct and breaking [[3.62822, 5.3249, 7.14642], [6.32493, 4.60408, 6.66067], [6.80794, 5.48752, 6.60977]]
Initial =,  [[[1.53403 6.96055 6.22653]
  [6.52341 5.45523 6.40056]
  [6.13581 6.27489 6.72929]]

 [[2.62822 5.3249  7.14642]
  [6.32493 4.60408 